# TD Methods with RBF & Function Approximation

TD(0) Q-Learning

In [ ]:
from __future__ import print_function, division
from builtins import range

import gymnasium as gym
#from gymnasium import wrappers

import os, sys
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
# from mpl_toolkits.mplot3d import Axes3D
from datetime import datetime
from time import time

'''
from sklearn.pipeline import FeatureUnion
from sklearn.preprocessing import StandardScaler
from sklearn.kernel_approximation import RBFSampler
'''
from sklearn.linear_model import SGDRegressor

from MountainCar_v0_common import plot_cost_to_go, FeatureTransformer, plot_running_avg, Model

# Common seed for reproducibility
SEED = 42
np.random.seed(SEED)
# Split data with fixed random_state
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
# Train a model with fixed random_state
# model = LogisticRegression(random_state=SEED, max_iter=200)
# Seed environment and action space
# obs, info = env.reset(seed=SEED)
# env.action_space.seed(SEED)

# SGDRegressor defaults:
# loss='squared_loss', penalty='l2', alpha=0.0001,
# l1_ratio=0.15, fit_intercept=True, n_iter=5, shuffle=True,
# verbose=0, epsilon=0.1, random_state=None, learning_rate='invscaling',
# eta0=0.01, power_t=0.25, warm_start=False, average=False

# returns a list of states_and_rewards, and the total reward
def run_episode(model, env, eps, gamma):
  observation = env.reset()[0]
  done = False
  totalreward = 0
    
  iters = 0
  while not done and iters < 10_000:
    action = model.sample_action(observation, eps)
      
    prev_observation = observation
    observation, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated

    # update the model
    if done:
      G = reward
    else:
      Qnext = model.predict(observation)
      # assert(next.shape == (1, env.action_space.n))
      G = reward + gamma*np.max(Qnext[0])

    model.update(prev_observation, action, G)

    totalreward += reward
    iters += 1

  return totalreward
    
if __name__ == '__main__':
  _start = time()
  env = gym.make('MountainCar-v0')
  ft = FeatureTransformer(env)
  # MODEL_CLASS = SGDRegressor # sklearn
  model = Model(env, ft, SGDRegressor, learning_rate="constant")
  gamma = 0.99
  '''
  if 'monitor' in sys.argv:
    filename = os.path.basename(__file__).split('.')[0]
    monitor_dir = './' + filename + '_' + str(datetime.now())
    env = wrappers.Monitor(env, monitor_dir)
  '''
  N = 300*2
  totalrewards = np.empty(N)
    
  for n in range(N):
    eps = 1.0/(0.1*n+1)
    # eps = 0.1*(0.97**n)
    if n == 199: print("eps:", eps)
    # eps = 1.0/np.sqrt(n+1)
    totalreward = run_episode(model, env, eps, gamma)
    totalrewards[n] = totalreward
    if (n + 1) % 10 == 0:
      print("\repisode:", n, "total reward:", totalreward, ' '*10, end="")
  print("avg reward for last 100 episodes:", totalrewards[-100:].mean())
  print("total steps:", -totalrewards.sum())
  print(f"Execution time: {time() - _start:.2f} seconds")    
  """
  plt.plot(totalrewards)
  plt.title("Rewards")
  plt.show()
  """
  plot_running_avg(totalrewards)

  # plot the optimal state-value function
  plot_cost_to_go(env, model)

## TD n-step

In [ ]:
from __future__ import print_function, division
from builtins import range
# Adapt Q-Learning script to use N-step method instead

import gymnasium as gym
# from gymnasium import wrappers
import os, sys
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

# replace SKLearn Regressor
# from sklearn.linear_model import SGDRegressor

class SGDRegressor:
  def __init__(self, **kwargs):
    self.w = None
    self.lr = 1e-2

  def partial_fit(self, X, Y):
    if self.w is None: # random initialization
        D = X.shape[1]
        self.w = np.random.randn(D) / np.sqrt(D)
    self.w += self.lr*(Y - X.dot(self.w)).dot(X)

  def predict(self, X):
    return X.dot(self.w)

# code we already wrote
# import q_learning # SGDRegressor
# from q_learning import plot_cost_to_go, FeatureTransformer, plot_running_avg # Model 
from MountainCar_v0_common import plot_cost_to_go, FeatureTransformer, plot_running_avg, Model

# Common seed for reproducibility
SEED = 42
np.random.seed(SEED)
# Split data with fixed random_state
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
# Train a model with fixed random_state
# model = LogisticRegression(random_state=SEED, max_iter=200)
# Seed environment and action space
# obs, info = env.reset(seed=SEED)
# env.action_space.seed(SEED)

# calculate everything up to max[Q(s,a)]
# Ex.
# R(t) + gamma*R(t+1) + ... + (gamma^(n-1))*R(t+n-1) + (gamma^n)*max[Q(s(t+n), a(t+n))]
# def calculate_return_before_prediction(rewards, gamma):
#   ret = 0
#   for r in reversed(rewards[1:]):
#     ret += r + gamma*ret
#   ret += rewards[0]
#   return ret

# returns a list of states_and_rewards, and the total reward
def run_episode(model, env, eps, gamma, n=5):
  observation = env.reset()[0]
  done = False
  totalreward = 0
  rewards, states, actions = [], [], []
  # array of [gamma^0, gamma^1, ..., gamma^(n-1)]
  multiplier = np.array([gamma]*n)**np.arange(n)
  
  iters = 0
  # while not done and iters < 200:
  while not done and iters < 10_000:
    # in earlier versions of gym, episode doesn't automatically
    # end when you hit 200 steps
    action = model.sample_action(observation, eps)

    states.append(observation)
    actions.append(action)

    prev_observation = observation
    observation, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated

    rewards.append(reward)

    # update the model
    if len(rewards) >= n:
      # return_up_to_prediction = calculate_return_before_prediction(rewards, gamma)
      return_up_to_prediction = multiplier.dot(rewards[-n:])
      action_values = model.predict(observation)[0]
      # print("action_values.shape:", action_values.shape)
      G = return_up_to_prediction + (gamma**n)*np.max(action_values)
      # print("G:", G)
      model.update(states[-n], actions[-n], G)

    # if len(rewards) > n:
    #   rewards.pop(0)
    #   states.pop(0)
    #   actions.pop(0)
    # assert(len(rewards) <= n)

    totalreward += reward
    iters += 1

  # empty the cache
  if n == 1:
    rewards = []
    states = []
    actions = []
  else:
    rewards = rewards[-n+1:]
    states = states[-n+1:]
    actions = actions[-n+1:]
  # unfortunately, new version of gym cuts you off at 200 steps
  # even if you haven't reached the goal.
  # it's not good to do this UNLESS you've reached the goal.
  # we are "really done" if position >= 0.5
  if observation[0] >= 0.5:
    # we actually made it to the goal
    # print("made it!")
    while len(rewards) > 0:
      G = multiplier[:len(rewards)].dot(rewards)
      model.update(states[0], actions[0], G)
      rewards.pop(0)
      states.pop(0)
      actions.pop(0)
  else:
    # we did not make it to the goal
    # print("didn't make it...")
    while len(rewards) > 0:
      guess_rewards = rewards + [-1]*(n - len(rewards))
      G = multiplier.dot(guess_rewards)
      model.update(states[0], actions[0], G)
      rewards.pop(0)
      states.pop(0)
      actions.pop(0)

  return totalreward


if __name__ == '__main__':
  _start = time()
  env = gym.make('MountainCar-v0')
  ft = FeatureTransformer(env)
  # MODEL_CLASS = SGDRegressor # custom
  model = Model(env, ft, SGDRegressor, learning_rate="constant")
  gamma = 0.99
  """
  if 'monitor' in sys.argv:
    filename = os.path.basename(__file__).split('.')[0]
    monitor_dir = './' + filename + '_' + str(datetime.now())
    env = wrappers.Monitor(env, monitor_dir)
  """
  N = 300*2
  totalrewards = np.empty(N)
  costs = np.empty(N)
  for n in range(N):
    # eps = 1.0/(0.1*n+1)
    eps = 0.1*(0.97**n)
    totalreward = run_episode(model, env, eps, gamma)
    totalrewards[n] = totalreward
    if (n + 1) % 10 == 0:
        print("\repisode:", n, "total reward:", totalreward, ' '*10, end="")
  print("avg reward for last 100 episodes:", totalrewards[-100:].mean())
  print("total steps:", -totalrewards.sum())
  print(f"Execution time: {time() - _start:.2f} seconds")    
  """
  plt.plot(totalrewards)
  plt.title("Rewards")
  plt.show()
  """
  plot_running_avg(totalrewards)

  # plot the optimal state-value function
  plot_cost_to_go(env, model)

## TD Lambda

In [ ]:
from __future__ import print_function, division
from builtins import range
# Adapt Q-Learning script to use TD(lambda) method instead

import gymnasium as gym
# from gymnasium import wrappers
import os, sys
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

# code we already wrote
from MountainCar_v0_common import plot_cost_to_go, FeatureTransformer, plot_running_avg # , Model
"""
gym_minor_version = int(gym.__version__.split('.')[1])
if gym_minor_version >= 19:
  exit("Please install OpenAI Gym 0.19.0 or earlier")
"""

class BaseModel:
  def __init__(self, D):
    self.w = np.random.randn(D) / np.sqrt(D)

  def partial_fit(self, input_, target, eligibility, lr=1e-2):
    self.w += lr*(target - input_.dot(self.w))*eligibility

  def predict(self, X):
    X = np.array(X)
    return X.dot(self.w)

# Holds one BaseModel for each action
class Model:
  def __init__(self, env, feature_transformer):
    self.env = env
    self.models = []
    self.feature_transformer = feature_transformer

    D = feature_transformer.dimensions
    self.eligibilities = np.zeros((env.action_space.n, D))

    for i in range(env.action_space.n):
      model = BaseModel(D)
      self.models.append(model)

  def predict(self, s):
    X = self.feature_transformer.transform([s])
    assert(len(X.shape) == 2)
    result = np.stack([m.predict(X) for m in self.models]).T
    assert(len(result.shape) == 2)
    return result

  def update(self, s, a, G, gamma, lambda_):
    X = self.feature_transformer.transform([s])
    assert(len(X.shape) == 2)
    self.eligibilities *= gamma*lambda_
    self.eligibilities[a] += X[0]
    self.models[a].partial_fit(X[0], G, self.eligibilities[a])

  def sample_action(self, s, eps):
    if np.random.random() < eps:
      return self.env.action_space.sample()
    else:
      return np.argmax(self.predict(s))


# returns a list of states_and_rewards, and the total reward
def run_episode(model, env, eps, gamma, lambda_):
  observation = env.reset()[0]
  done = False
  totalreward = 0
  iters = 0
  # while not done and iters < 200:
  while not done and iters < 10_000:
    action = model.sample_action(observation, eps)
    prev_observation = observation
    observation, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated
      
    # update the model
    Qnext = model.predict(observation)
    assert(Qnext.shape == (1, env.action_space.n))
    G = reward + gamma*np.max(Qnext[0])
    model.update(prev_observation, action, G, gamma, lambda_)

    totalreward += reward
    iters += 1

  return totalreward


if __name__ == '__main__':
  _start = time()
  env = gym.make('MountainCar-v0')
  ft = FeatureTransformer(env)
  model = Model(env, ft)
  gamma = 0.9999
  lambda_ = 0.7
  '''
  if 'monitor' in sys.argv:
    filename = os.path.basename(__file__).split('.')[0]
    monitor_dir = './' + filename + '_' + str(datetime.now())
    env = wrappers.Monitor(env, monitor_dir)
  '''
  N = 300*2
  totalrewards = np.empty(N)
  costs = np.empty(N)
  for n in range(N):
    # eps = 1.0/(0.1*n+1)
    eps = 0.1*(0.97**n)
    # eps = 0.5/np.sqrt(n+1)
    totalreward = run_episode(model, env, eps, gamma, lambda_)
    totalrewards[n] = totalreward
    if (n + 1) % 10 == 0:
      print("\repisode:", n, "total reward:", totalreward, ' '*10, end="")
  print("avg reward for last 100 episodes:", totalrewards[-100:].mean())
  print("total steps:", -totalrewards.sum())
  print(f"Execution time: {time() - _start:.2f} seconds")    
  """
  plt.plot(totalrewards)
  plt.title("Rewards")
  plt.show()
  """
  plot_running_avg(totalrewards)

  # plot the optimal state-value function
  plot_cost_to_go(env, model)